In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# These classes are assumed to be defined in your environment
from ChromaVDB.chroma import ChromaFramework
from DeepGraphDB import DeepGraphDB

gdb = DeepGraphDB()
gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")
vdb = ChromaFramework(persist_directory="./ChromaVDB/chroma_db")

records = vdb.list_records()

names = [record['name'] for record in records if record['embedding_type'] == 'graph']
entities = [record['entity'] for record in records if record['embedding_type'] == 'graph']
graph_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'graph']
text_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'text']
ids = [record['id'] for record in records if record['embedding_type'] == 'graph']

# Load and filter patient data
# data = pd.read_csv("/home/cc/PHD/dglframework/cptac/patient_gene_matrix_BRCA.csv", low_memory=False)
# # data = data[(data['site_of_resection_or_biopsy'] == 'Breast, NOS') & (data['primary_diagnosis'].isin(['Infiltrating duct carcinoma, NOS','Lobular carcinoma, NOS']))]
# data = data[(data['site_of_resection_or_biopsy'] == 'Breast, NOS') & (data['primary_diagnosis'] == 'Infiltrating duct carcinoma, NOS' )]

data = pd.read_excel('data/2025_03_29.xlsx') # (Diffuse Large B-cell Lymphoma)

# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
metaboliti = list(data.columns[249:530])

meta_embs = model.encode(metaboliti, show_progress_bar=True)

In [ ]:
res = vdb.search_records(
    # query=meta_embs[0],
    query=model.encode('Diffuse Large B-cell Lymphoma'),
    embedding_type='text',
    n_results=10,
    #entity='exposure',
)

In [ ]:
vid = '8749ac00-1efc-4bb4-94d3-6d9ba7bbe9a7'

key = next((k for k, v in vdb.global_to_vids_mapping.items() if v == vid), None)